# Silver Layer — SCD Type 2 (Yelp Users)

## Logic MERGE INTO
Detect thay đổi bằng cách so sánh `review_count`, `fans`, `average_stars` với bản ghi active hiện tại.

**Khi có thay đổi:**
1. **BƯỚC 1**: Đóng record cũ — set `is_current = false`, `end_time = now()`
2. **BƯỚC 2**: Insert record mới — `is_current = true`, `effective_time = now()`, `end_time = NULL`

**Khi không có thay đổi:** bỏ qua (idempotent)

**Khi user mới (chưa tồn tại):** Insert trực tiếp

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Yelp_Silver_SCD2") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.io.ResolvingFileIO") \
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .config("spark.sql.defaultCatalog", "nessie") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

hc = spark.sparkContext._jsc.hadoopConfiguration()
hc.set("fs.s3a.endpoint",          "http://minio:9000")
hc.set("fs.s3a.access.key",        "admin")
hc.set("fs.s3a.secret.key",        "password")
hc.set("fs.s3a.path.style.access", "true")
hc.set("fs.s3a.connection.ssl.enabled", "false")
hc.set("fs.s3a.impl",              "org.apache.hadoop.fs.s3a.S3AFileSystem")
hc.set("fs.s3a.aws.credentials.provider",
       "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

spark.sparkContext.setLogLevel("ERROR")
print("✅ SparkSession ready")

✅ SparkSession ready


26/06/16 01:15:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
from pyspark.sql.functions import col, current_timestamp

# Stop stream cũ
for q in spark.streams.active:
    q.stop()
    print(f"⏹ Stopped: {q.name}")

# Tạo namespace
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")

# Tạo bảng Silver SCD2
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.silver.yelp_users_scd2 (
        user_id        STRING,
        name           STRING,
        review_count   BIGINT,
        yelping_since  STRING,
        useful         BIGINT,
        funny          BIGINT,
        cool           BIGINT,
        fans           BIGINT,
        average_stars  DOUBLE,
        elite          STRING,
        -- SCD2 metadata
        is_current     BOOLEAN,
        effective_time TIMESTAMP,
        end_time       TIMESTAMP
    ) USING iceberg
    TBLPROPERTIES (
        'write.merge.mode' = 'copy-on-write',
        'write.update.mode' = 'copy-on-write'
    )
""")

print("✅ Table nessie.silver.yelp_users_scd2 ready")

✅ Table nessie.silver.yelp_users_scd2 ready


In [3]:
def process_silver_scd2(micro_batch_df, batch_id):
    if micro_batch_df.isEmpty():
        return

    # Dedup trong cùng micro-batch — giữ record cuối cùng của mỗi user_id
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number

    w = Window.partitionBy("user_id").orderBy(col("review_count").desc())
    dedup_df = micro_batch_df \
        .withColumn("_rn", row_number().over(w)) \
        .filter(col("_rn") == 1) \
        .drop("_rn")

    dedup_df.createOrReplaceGlobalTempView("yelp_user_updates")

    # ── BƯỚC 1: Đóng record cũ khi có thay đổi ──────────────────
    # Detect thay đổi ở 3 field quan trọng nhất
    spark.sql("""
        MERGE INTO nessie.silver.yelp_users_scd2 AS target
        USING (
            SELECT u.*
            FROM global_temp.yelp_user_updates u
            JOIN nessie.silver.yelp_users_scd2 t
              ON u.user_id = t.user_id
             AND t.is_current = true
             AND (
                 u.review_count  != t.review_count
              OR u.fans          != t.fans
              OR u.average_stars != t.average_stars
             )
        ) AS source
        ON target.user_id = source.user_id
           AND target.is_current = true
        WHEN MATCHED THEN
            UPDATE SET
                target.is_current = false,
                target.end_time   = current_timestamp()
    """)

    # ── BƯỚC 2: Insert record mới (user mới HOẶC vừa được đóng) ─
    spark.sql("""
        MERGE INTO nessie.silver.yelp_users_scd2 AS target
        USING (
            SELECT u.*
            FROM global_temp.yelp_user_updates u
            LEFT JOIN nessie.silver.yelp_users_scd2 t
              ON u.user_id = t.user_id
             AND t.is_current = true
            WHERE t.user_id IS NULL
        ) AS source
        ON target.user_id = source.user_id
           AND target.is_current = true
        WHEN NOT MATCHED THEN
            INSERT (
                user_id, name, review_count, yelping_since,
                useful, funny, cool, fans, average_stars, elite,
                is_current, effective_time, end_time
            )
            VALUES (
                source.user_id, source.name, source.review_count,
                source.yelping_since, source.useful, source.funny,
                source.cool, source.fans, source.average_stars,
                source.elite,
                true, current_timestamp(), NULL
            )
    """)


# Đọc stream từ Bronze
bronze_stream = spark.readStream \
    .format("iceberg") \
    .load("nessie.bronze.yelp_users")

print("[*] Khởi chạy Silver SCD2 stream...")

silver_query = bronze_stream.writeStream \
    .foreachBatch(process_silver_scd2) \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://warehouse/checkpoints/silver_yelp_users_scd2") \
    .trigger(processingTime="20 seconds") \
    .queryName("silver_yelp_users_scd2") \
    .start()

print("[+] Silver SCD2 stream đang chạy!")

[*] Khởi chạy Silver SCD2 stream...
[+] Silver SCD2 stream đang chạy!


In [13]:
# Monitor SCD2 — xem tỷ lệ active vs expired
import time

for i in range(1):
    try:
        total   = spark.sql("SELECT COUNT(*) FROM nessie.silver.yelp_users_scd2").collect()[0][0]
        active  = spark.sql("SELECT COUNT(*) FROM nessie.silver.yelp_users_scd2 WHERE is_current = true").collect()[0][0]
        expired = total - active
        streams = len(spark.streams.active)
        print(f"[{i+1:02d}] total={total:>10,} | active={active:>10,} | expired={expired:>8,} | streams={streams}")
    except Exception as e:
        print(f"[{i+1:02d}] Error: {e}")
    time.sleep(20)

[01] total= 2,783,056 | active= 1,987,898 | expired= 795,158 | streams=1


In [10]:
# ============================================================
# VALIDATION — Chứng minh SCD2 hoạt động
# ============================================================

print("=" * 60)
print("VALIDATION: Users có lịch sử thay đổi (version > 1)")
print("=" * 60)

changed = spark.sql("""
    SELECT
        user_id,
        COUNT(*) AS version_count,
        MIN(effective_time) AS first_seen,
        MAX(effective_time) AS last_changed
    FROM nessie.silver.yelp_users_scd2
    GROUP BY user_id
    HAVING COUNT(*) > 1
    ORDER BY version_count DESC
    LIMIT 10
""")

changed.show(truncate=False)
count = changed.count()
print(f"→ Tìm thấy {count:,} users có lịch sử thay đổi")

if count > 0:
    print("\n✅ SCD Type 2 hoạt động đúng — có UPDATE history!")
else:
    print("\n⚠️  Chưa có history — Silver stream chưa xử lý xong Pass 2/3")

VALIDATION: Users có lịch sử thay đổi (version > 1)


+----------------------+-------------+--------------------------+--------------------------+
|user_id               |version_count|first_seen                |last_changed              |
+----------------------+-------------+--------------------------+--------------------------+
|0M8hxXt5n6WTlIyM9Ue9zg|3            |2026-06-16 01:20:03.023675|2026-06-16 02:05:23.469138|
|2Lxd9LnB0KfjgTlQEiDs6w|3            |2026-06-16 01:21:22.292569|2026-06-16 02:05:23.469138|
|0mMPK2qRCbqopyqxVSmQRg|3            |2026-06-16 01:19:22.746901|2026-06-16 02:05:23.469138|
|2TDXQDETUD0XPek3LOSf2Q|3            |2026-06-16 01:20:03.023675|2026-06-16 02:05:23.469138|
|-HHfUGxD6PIJPGTOUNgchA|3            |2026-06-16 01:19:22.746901|2026-06-16 02:05:23.469138|
|3V2Lp3lIAOABEa5l6IJ50g|3            |2026-06-16 01:17:41.745881|2026-06-16 02:05:23.469138|
|20HHEvSjTvUQeQbEyfon9w|3            |2026-06-16 01:19:22.746901|2026-06-16 02:05:23.469138|
|3oP-HNJxTNVc4er4naXRGg|3            |2026-06-16 01:17:24.038016|2026-

→ Tìm thấy 10 users có lịch sử thay đổi

✅ SCD Type 2 hoạt động đúng — có UPDATE history!


In [11]:
# Full history của 1 user cụ thể (chọn user có nhiều version nhất)

top_user = spark.sql("""
    SELECT user_id, COUNT(*) as v
    FROM nessie.silver.yelp_users_scd2
    GROUP BY user_id
    ORDER BY v DESC
    LIMIT 1
""").collect()[0]["user_id"]

print(f"User có nhiều version nhất: {top_user}")
print()

spark.sql(f"""
    SELECT
        user_id,
        review_count,
        fans,
        average_stars,
        is_current,
        effective_time,
        end_time
    FROM nessie.silver.yelp_users_scd2
    WHERE user_id = '{top_user}'
    ORDER BY effective_time ASC
""").show(truncate=False)

User có nhiều version nhất: 39dVyJb0aQ14_HphqwQfkQ



+----------------------+------------+----+-------------+----------+--------------------------+--------------------------+
|user_id               |review_count|fans|average_stars|is_current|effective_time            |end_time                  |
+----------------------+------------+----+-------------+----------+--------------------------+--------------------------+
|39dVyJb0aQ14_HphqwQfkQ|20          |0   |3.76         |false     |2026-06-16 01:18:41.611962|2026-06-16 01:59:20.228047|
|39dVyJb0aQ14_HphqwQfkQ|26          |3   |3.82         |false     |2026-06-16 01:59:29.305267|2026-06-16 02:07:40.181532|
|39dVyJb0aQ14_HphqwQfkQ|36          |4   |3.79         |true      |2026-06-16 02:07:42.93405 |NULL                      |
+----------------------+------------+----+-------------+----------+--------------------------+--------------------------+



In [12]:
# Iceberg Time Travel — xem snapshot tại thời điểm Pass 1 (chỉ có initial load)

snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM nessie.silver.yelp_users_scd2.snapshots
    ORDER BY committed_at
""")

snapshots.show(truncate=False)

snap_list = snapshots.collect()
if len(snap_list) >= 2:
    first_snap = snap_list[0]["snapshot_id"]
    print(f"\nTime travel → snapshot đầu tiên (Pass 1): {first_snap}")
    count_v1 = spark.sql(f"""
        SELECT COUNT(*) as cnt
        FROM nessie.silver.yelp_users_scd2 VERSION AS OF {first_snap}
    """).collect()[0]["cnt"]
    count_now = spark.sql("SELECT COUNT(*) as cnt FROM nessie.silver.yelp_users_scd2").collect()[0]["cnt"]
    print(f"  Records tại snapshot 1 : {count_v1:,}")
    print(f"  Records hiện tại       : {count_now:,}")
    print(f"  Tăng thêm              : {count_now - count_v1:,} (đây là SCD2 history records)")

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|5912976874437727619|2026-06-16 01:16:24.542|overwrite|
|361414564735491513 |2026-06-16 01:16:26.309|append   |
|101107265991142459 |2026-06-16 01:16:42.633|overwrite|
|3600751820086338538|2026-06-16 01:16:44.489|append   |
|5711011580776692283|2026-06-16 01:17:02.409|overwrite|
|7167056469446899184|2026-06-16 01:17:03.682|append   |
|4126924986492364422|2026-06-16 01:17:23.949|overwrite|
|2702554889069968106|2026-06-16 01:17:26.108|append   |
|2539487136036375886|2026-06-16 01:17:41.649|overwrite|
|3613280388974412248|2026-06-16 01:17:43.212|append   |
|8970602758181496220|2026-06-16 01:18:02.421|overwrite|
|1815378533041842886|2026-06-16 01:18:04.202|append   |
|5032261615102457855|2026-06-16 01:18:22.208|overwrite|
|5015972982831717934|2026-06-16 01:18:24.328|append   |
|864005344751511726 |2026-06-16 01:18:41.545|ove